In [ ]:
# ===========================
# PERMANENT COLAB SETUP
# Mount Google Drive + Set Working Directory
# ===========================

from google.colab import drive
import os   # ✅ this is the missing import!

# 1️⃣ Mount Google Drive
drive.mount('/content/drive')

# 2️⃣ Define and create your project folder inside Drive
WORKDIR = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

# 3️⃣ Verify it’s working
print("✅ Drive mounted successfully!")
print("📂 Working directory set to:", os.getcwd())


In [ ]:
# data handling and preprocessing
# Step 0: Imports
# ==========================
import pandas as pd
import re
import numpy as np
from tqdm.auto import tqdm
tqdm.pandas()

# Use DistilBERT if available
try:
    from sentence_transformers import SentenceTransformer
    use_bert = True
except:
    from sklearn.feature_extraction.text import TfidfVectorizer
    use_bert = False

# ==========================
# Step 1: Load datasets
# ==========================
image_dir = '/content/images'
train_file = '/content/train (1).csv'
test_file = '/content/test.csv'

df_train = pd.read_csv(train_file)
df_test = pd.read_csv(test_file)

text_col = 'catalog_content'  # Change if your column has a different name

# ==========================
# Step 2: Fast IPQ Extraction
# ==========================
ipq_pattern = re.compile(r'(?i)(?:pack(?: of)?|IPQ:?)\s*(\d+)')

def extract_ipq(text):
    if pd.isna(text):
        return 0
    match = ipq_pattern.search(text)
    return int(match.group(1)) if match else 0

for df in [df_train, df_test]:
    df['IPQ'] = df[text_col].progress_apply(extract_ipq)

# ==========================
# Step 3: Basic Text Features
# ==========================
for df in [df_train, df_test]:
    df['title_length'] = df[text_col].progress_apply(lambda x: len(str(x)) if pd.notna(x) else 0)
    df['token_count']  = df[text_col].progress_apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)

# ==========================
# Step 4: Text Embeddings (Fast & High-quality)
# ==========================
if use_bert:
    print("Using DistilBERT embeddings for high accuracy & speed...")
    model = SentenceTransformer('distilbert-base-nli-mean-tokens')

    # Encode in batches for speed & memory efficiency
    df_train_embeddings = model.encode(df_train[text_col].fillna("").tolist(),
                                       batch_size=64,
                                       show_progress_bar=True)
    df_test_embeddings  = model.encode(df_test[text_col].fillna("").tolist(),
                                       batch_size=64,
                                       show_progress_bar=True)
else:
    print("BERT not available. Using TF-IDF fallback (fast & scalable).")
    tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=50000)
    combined_text = pd.concat([df_train[text_col].fillna(""), df_test[text_col].fillna("")])
    tfidf.fit(combined_text)

    df_train_embeddings = tfidf.transform(df_train[text_col].fillna("")).toarray()
    df_test_embeddings  = tfidf.transform(df_test[text_col].fillna("")).toarray()

# ==========================
# Step 5: Quick Check
# ==========================
print("Train df shape:", df_train.shape, "Embeddings shape:", df_train_embeddings.shape)
print("Test df shape:", df_test.shape, "Embeddings shape:", df_test_embeddings.shape)

# ✅ df_train and df_test now have:
#   - IPQ (integer)
#   - title_length, token_count
#   - Text embeddings (768-d for DistilBERT, 50k-d for TF-IDF fallback)


In [ ]:
# Step 2: Image Downloading and Handling
import os
import pandas as pd
import requests
from PIL import Image
from io import BytesIO
import time # Import time for potential delays
import numpy as np # Import numpy for potential use in data handling

image_dir = '/content/images'
os.makedirs(image_dir, exist_ok=True)

# Helper function to download images (from src/utils.py - simulated)
def download_images(df, image_dir, dataset_name):
    print(f"Downloading images for {dataset_name} dataset...")
    # Filter out rows with missing or invalid image URLs early
    df_with_images = df[df['image_url'].notna() & df['image_url'].str.startswith('http')].copy()

    # Add a simple progress indicator
    total_images = len(df_with_images)
    downloaded_count = 0
    skipped_count = 0
    error_count = 0

    print(f"Attempting to download {total_images} images for {dataset_name}...")

    for index, row in df_with_images.iterrows():
        sample_id = row['sample_id']
        image_url = row['image_url']
        img_path = os.path.join(image_dir, f"{sample_id}.jpg")

        # Skip download if image already exists
        if os.path.exists(img_path):
            skipped_count += 1
        else:
            try:
                response = requests.get(image_url, timeout=10)
                response.raise_for_status() # Raise an exception for bad status codes
                img = Image.open(BytesIO(response.content)).convert('RGB')
                img.save(img_path)
                downloaded_count += 1
            except requests.exceptions.Timeout:
                # print(f"Timeout downloading image for sample_id {sample_id}")
                error_count += 1
                pass # Handle timeout
            except requests.exceptions.RequestException as e:
                # print(f"Request error downloading image for sample_id {sample_id}: {e}")
                error_count += 1
                pass # Handle other request errors
            except Exception as e:
                # print(f"Error processing downloaded image for sample_id {sample_id}: {e}")
                error_count += 1
                pass # Handle image processing errors

        if (downloaded_count + skipped_count + error_count) % 100 == 0:
             print(f"Processed {downloaded_count + skipped_count + error_count}/{total_images} images ({downloaded_count} downloaded, {skipped_count} skipped, {error_count} errors)...")

        # Optional: Add a small delay to avoid overwhelming the server
        # time.sleep(0.05) # Reduced delay

    print(f"\nFinished processing images for {dataset_name}.")
    print(f"Total images with valid URLs: {total_images}")
    print(f"Successfully downloaded: {downloaded_count}")
    print(f"Skipped (already exists): {skipped_count}")
    print(f"Errors during download/processing: {error_count}")


# Assuming train and test dataframes are loaded in a previous cell
if 'image_link' in train.columns:
    # Assuming 'image_link' is the column name for image URLs
    train = train.rename(columns={'image_link': 'image_url'})
    download_images(train, image_dir, "train")
else:
    print("Warning: 'image_link' column not found in train dataframe. Skipping image download for train.")

if 'image_link' in test.columns:
    # Assuming 'image_link' is the column name for image URLs
    test = test.rename(columns={'image_link': 'image_url'})
    download_images(test, image_dir, "test")
else:
     print("Warning: 'image_link' column not found in test dataframe. Skipping image download for test.")


In [ ]:
# Separate Images into Train/Test Subdirectories
#Step 1: Make train and test folders
!mkdir -p /content/images/train /content/images/test
#step-2
import os
import pandas as pd
import shutil

# File paths
image_dir = '/content/images' # The directory where all images are currently
train_file = '/content/train (1).csv'
test_file = '/content/test.csv'
train_image_dir = os.path.join(image_dir, 'train')
test_image_dir = os.path.join(image_dir, 'test')

# Create the subdirectories if they don't exist (User confirmed this is done)
# os.makedirs(train_image_dir, exist_ok=True)
# os.makedirs(test_image_dir, exist_ok=True)

# Load the train and test dataframes to get sample IDs
# Assuming they are already loaded, but loading them again for this specific task
try:
    train = pd.read_csv(train_file, dtype={'sample_id': str}, engine='python', on_bad_lines='skip')
    test = pd.read_csv(test_file, dtype={'sample_id': str}, engine='python', on_bad_lines='skip')
    print("Train and test dataframes loaded for separating images.")
except FileNotFoundError as e:
    print(f"Error loading file: {e}. Please ensure the dataset files are in the correct '/content/' directory.")
    # Exit or handle the error appropriately if dataframes can't be loaded
    # exit() # Or raise an exception
    train = pd.DataFrame(columns=['sample_id']) # Create empty dataframe to prevent errors
    test = pd.DataFrame(columns=['sample_id'])


# Get the list of sample IDs for training and testing
train_sample_ids = set(train['sample_id'].tolist())
test_sample_ids = set(test['sample_id'].tolist())

print(f"Found {len(train_sample_ids)} train sample IDs and {len(test_sample_ids)} test sample IDs.")

# Iterate through all files in the main image directory
print(f"Separating images in {image_dir}...")
moved_count = 0
skipped_count = 0
not_found_count = 0

for filename in os.listdir(image_dir):
    src_path = os.path.join(image_dir, filename)

    # Skip directories (like train/test folders)
    if os.path.isdir(src_path):
        continue

    # Remove extension to match CSV ids
    file_id = os.path.splitext(filename)[0]

    if file_id in train_sample_ids:
        dest_path = os.path.join(train_image_dir, filename)
        # Move the file if it's not already in the target directory
        if not os.path.exists(dest_path):
            try:
                shutil.move(src_path, dest_path)
                moved_count += 1
            except Exception as e:
                print(f"Error moving file {filename} to train: {e}")
                not_found_count += 1 # Count as not processed successfully
        else:
            skipped_count += 1 # File already in destination
    elif file_id in test_sample_ids:
        dest_path = os.path.join(test_image_dir, filename)
         # Move the file if it's not already in the target directory
        if not os.path.exists(dest_path):
            try:
                shutil.move(src_path, dest_path)
                moved_count += 1
            except Exception as e:
                 print(f"Error moving file {filename} to test: {e}")
                 not_found_count += 1 # Count as not processed successfully
        else:
            skipped_count += 1 # File already in destination
    else:
        # Count images that don't match train or test IDs
        not_found_count += 1
        # Optional: print(f"Warning: Image {filename} has sample_id not found in train or test dataframes.")


print(f"\nFinished separating images.")
print(f"Moved {moved_count} images.")
print(f"Skipped {skipped_count} images (already in target folder).")
print(f"Images not matched to train/test IDs or with errors: {not_found_count}")
print(f"Total files processed in image directory (excluding subfolders): {moved_count + skipped_count + not_found_count}")

# Note: This script moves the files. If you want to copy instead, use shutil.copy2()

In [ ]:
# ==========================
# FEATURE ENGG - TEXT - for feature csv files
# ==========================
import re
import numpy as np
import pandas as pd
import os

# Load datasets
train_file = '/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/train (1).csv'  # Your raw input file
test_file  = '/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/test.csv'

train = pd.read_csv(train_file)
test  = pd.read_csv(test_file)

# Safe float conversion
def safe_float(s):
    try:
        return float(s)
    except:
        return np.nan

# Parse catalog_content to extract rich features
def parse_catalog_content(text):
    text = str(text)
    data = {
        'item_name': None,
        'description': None,
        'bullet_points': None,
        'value': np.nan,
        'unit': None,
        'is_gluten_free': 0,
        'is_non_gmo': 0,
        'is_sugar_free': 0,
        'is_kosher': 0,
        'is_organic': 0,
        'pack_size': np.nan,
        'item_size_oz': np.nan
    }

    # 1️⃣ Item Name
    m = re.search(r'Item Name:\s*(.*?)(?:,|\n|$)', text)
    if m: data['item_name'] = m.group(1).strip()

    # 2️⃣ Description
    m = re.search(r'Product Description:\s*(.*?)(?:Bullet Point:|Value:|Unit:|Item Name:|\n\n|$)', text, re.DOTALL)
    if m: data['description'] = m.group(1).strip()

    # 3️⃣ Bullet Points
    matches = re.findall(r'Bullet Point:\s*(.*?)(?:$|\n)', text, re.MULTILINE)
    if matches: data['bullet_points'] = " ".join([bp.strip() for bp in matches])

    # 4️⃣ Value
    m = re.search(r'Value:\s*([\d\.]+)', text)
    if m: data['value'] = safe_float(m.group(1))

    # 5️⃣ Unit
    m = re.search(r'Unit:\s*(\w+)', text)
    if m: data['unit'] = m.group(1).strip()

    # 6️⃣ Flags
    if re.search(r'gluten\s*free', text, re.IGNORECASE): data['is_gluten_free'] = 1
    if re.search(r'non-?gmo', text, re.IGNORECASE): data['is_non_gmo'] = 1
    if re.search(r'sugar\s*free', text, re.IGNORECASE): data['is_sugar_free'] = 1
    if re.search(r'kosher', text, re.IGNORECASE): data['is_kosher'] = 1
    if re.search(r'organic', text, re.IGNORECASE): data['is_organic'] = 1

    # 7️⃣ Pack Size
    m = re.search(r'Pack of\s*(\d+)', text, re.IGNORECASE)
    if m: data['pack_size'] = safe_float(m.group(1))

    # 8️⃣ Item Size Oz
    m = re.search(r'([\d\.]+)\s*(?:Ounces|Oz)', text, re.IGNORECASE)
    if m: data['item_size_oz'] = safe_float(m.group(1))

    return data


# Apply parser
print("⏳ Extracting rich tabular features from catalog_content ...")
parsed_train = train['catalog_content'].apply(parse_catalog_content).apply(pd.Series)
parsed_test  = test['catalog_content'].apply(parse_catalog_content).apply(pd.Series)

# Merge back
train_features = pd.concat([train, parsed_train], axis=1)
test_features  = pd.concat([test, parsed_test], axis=1)

print("✅ All rich tabular features extracted successfully!\n")
print("Train shape:", train_features.shape)
print("Test shape:", test_features.shape)

# Show preview
print(train_features.head(3)[['item_name','value','unit','is_gluten_free','pack_size','item_size_oz']])

# Save for next step
os.makedirs("feature_data", exist_ok=True)
train_features.to_csv("feature_data/train_features.csv", index=False)
test_features.to_csv("feature_data/test_features.csv", index=False)

print("\n💾 Saved:")
print(" - feature_data/train_features.csv")
print(" - feature_data/test_features.csv")

# if any error :
# Save as CSV
train.to_csv('train_features.csv', index=False)
test.to_csv('test_features.csv', index=False)





its output : ⏳ Extracting rich tabular features from catalog_content ...
✅ All rich tabular features extracted successfully!

Train shape: (75000, 16)
Test shape: (75000, 15)
                           item_name  value   unit  is_gluten_free  pack_size  \
0  La Victoria Green Taco Sauce Mild   72.0     Fl               0        6.0   
1                    Salerno Cookies   32.0  Ounce               0        4.0   
2        Bear Creek Hearty Soup Bowl   11.4  Ounce               0        6.0   

   item_size_oz  
0           NaN  
1           NaN  
2           NaN  

💾 Saved:
 - feature_data/train_features.csv
 - feature_data/test_features.csv

In [ ]:
!pip install timm

# ============================================================
# IMAGE FEATURE EXTRACTION - EfficientNet-B3 (Robust & Rich)
# ============================================================

import os
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms
import timm
from tqdm.auto import tqdm

# ============================================================
# FILE PATHS
# ============================================================
image_dir = '/content/images'
train_image_dir = os.path.join(image_dir, 'train')
test_image_dir = os.path.join(image_dir, 'test')

# Ensure train/test dataframes are loaded
# train, test = pd.read_csv(...)

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 1️⃣ Load EfficientNet-B3 pretrained
# ============================================================
# num_classes=0 returns feature vector instead of classification
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
model.eval()
model.to(device)

# ============================================================
# 2️⃣ Preprocessing
# ============================================================
preprocess = transforms.Compose([
    transforms.Resize(300),
    transforms.CenterCrop(300),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# ============================================================
# 3️⃣ Safe float conversion
# ============================================================
def safe_float(s):
    try:
        return float(s)
    except:
        return np.nan

# ============================================================
# 4️⃣ Feature extraction function (robust)
# ============================================================
def extract_image_features(sample_id, base_dir, dataset_name):
    img_path = os.path.join(base_dir, dataset_name, f"{sample_id}.jpg")

    # Zero vector based on model output
    dummy_input = torch.zeros(1, 3, 300, 300).to(device)
    with torch.no_grad():
        zero_vector = model(dummy_input).detach().cpu().numpy().squeeze()
        zero_vector = zero_vector / np.linalg.norm(zero_vector) if np.linalg.norm(zero_vector) > 0 else zero_vector

    if os.path.exists(img_path):
        try:
            img = Image.open(img_path).convert('RGB')
            img_t = preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad():
                feats = model(img_t).detach().cpu().numpy().squeeze()
            feats = feats / np.linalg.norm(feats) if np.linalg.norm(feats) > 0 else feats
            return feats
        except:
            return zero_vector
    else:
        return zero_vector

# ============================================================
# 5️⃣ Extract embeddings for train/test
# ============================================================
print("Extracting rich image embeddings for TRAIN dataset...")
train_image_embeds = np.vstack([
    extract_image_features(sid, image_dir, 'train') for sid in tqdm(train['sample_id'])
])

print("Extracting rich image embeddings for TEST dataset...")
test_image_embeds = np.vstack([
    extract_image_features(sid, image_dir, 'test') for sid in tqdm(test['sample_id'])
])

print("✅ Rich Image Embeddings extraction completed.")
print("Train embeddings shape:", train_image_embeds.shape)
print("Test embeddings shape:", test_image_embeds.shape)


In [ ]:
#TEXT FEATURE ENGINEERING - (for embeddings)
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd

train_features = pd.read_csv("/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/feature_data/train_features.csv")
test_features  = pd.read_csv("/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/feature_data/test_features.csv")


train_texts = (train_features['description'].fillna('') + " " + train_features['bullet_points'].fillna('')).tolist()
test_texts  = (test_features['description'].fillna('') + " " + test_features['bullet_points'].fillna('')).tolist()

# Use a fast model; 'all-MiniLM-L6-v2' is lightweight and good for tabular tasks
model = SentenceTransformer('all-MiniLM-L6-v2')

print("⏳ Generating text embeddings ...")
df_train_embeddings = model.encode(train_texts, show_progress_bar=True, batch_size=64)
df_test_embeddings  = model.encode(test_texts, show_progress_bar=True, batch_size=64)

print("✅ Text embeddings generated.")
print("Train embeddings shape:", df_train_embeddings.shape)
print("Test embeddings shape:", df_test_embeddings.shape)

np.save("/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/feature_data/train_text_embeddings.npy", df_train_embeddings)
np.save("/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/feature_data/test_text_embeddings.npy", df_test_embeddings)
print("✅ Saved text embeddings to feature_data folder")



its output : /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning:
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
modules.json: 100%
 349/349 [00:00<00:00, 30.0kB/s]
config_sentence_transformers.json: 100%
 116/116 [00:00<00:00, 11.3kB/s]
README.md:
 10.5k/? [00:00<00:00, 1.06MB/s]
sentence_bert_config.json: 100%
 53.0/53.0 [00:00<00:00, 5.79kB/s]
config.json: 100%
 612/612 [00:00<00:00, 60.9kB/s]
model.safetensors: 100%
 90.9M/90.9M [00:01<00:00, 97.0MB/s]
tokenizer_config.json: 100%
 350/350 [00:00<00:00, 26.0kB/s]
vocab.txt:
 232k/? [00:00<00:00, 6.29MB/s]
tokenizer.json:
 466k/? [00:00<00:00, 18.9MB/s]
special_tokens_map.json: 100%
 112/112 [00:00<00:00, 5.86kB/s]
config.json: 100%
 190/190 [00:00<00:00, 9.07kB/s]
⏳ Generating text embeddings ...
Batches: 100%
 1172/1172 [38:06<00:00, 14.95it/s]
Batches: 100%
 1172/1172 [38:40<00:00, 10.89it/s]
✅ Text embeddings generated.
Train embeddings shape: (75000, 384)
Test embeddings shape: (75000, 384)

In [ ]:
# ==========================
# STEP 5: TABULAR FEATURE ENGINEERING - RICH FEATURES
# ==========================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import os

# --------------------------
# 0. Load RAW CSVs directly to ensure full dataset is used
# --------------------------
# Assuming raw data is in this directory as per previous cells
raw_data_directory = "/content/drive/MyDrive/amazon ML"

train_file_raw = os.path.join(raw_data_directory, 'train (1).csv')
test_file_raw  = os.path.join(raw_data_directory, 'test.csv')

train = pd.read_csv(train_file_raw)
test  = pd.read_csv(test_file_raw) # Use test for tabular FE, test_raw will be used for submission

print("✅ Raw CSVs loaded successfully for Tabular Feature Engineering!")
print("Train shape:", train.shape)
print("Test shape:", test.shape)

# Re-extract rich features from catalog_content since we are loading raw data
# This part was originally in Text FE, but needed here if loading raw CSVs
import re

def safe_float(s):
    try:
        return float(s)
    except:
        return np.nan

def parse_catalog_content(text):
    text = str(text)
    data = {
        'item_name': None,
        'description': None,
        'bullet_points': None,
        'value': np.nan,
        'unit': None,
        'is_gluten_free': 0,
        'is_non_gmo': 0,
        'is_sugar_free': 0,
        'is_kosher': 0,
        'is_organic': 0,
        'pack_size': np.nan,
        'item_size_oz': np.nan
    }

    m = re.search(r'Item Name:\s*(.*?)(?:,|\n|$)', text)
    if m: data['item_name'] = m.group(1).strip()
    m = re.search(r'Product Description:\s*(.*?)(?:Bullet Point:|Value:|Unit:|Item Name:|\n\n|$)', text, re.DOTALL)
    if m: data['description'] = m.group(1).strip()
    matches = re.findall(r'Bullet Point:\s*(.*?)(?:$|\n)', text, re.MULTILINE)
    if matches: data['bullet_points'] = " ".join([bp.strip() for bp in matches])
    m = re.search(r'Value:\s*([\d\.]+)', text)
    if m: data['value'] = safe_float(m.group(1))
    m = re.search(r'Unit:\s*(\w+)', text)
    if m: data['unit'] = m.group(1).strip()
    if re.search(r'gluten\s*free', text, re.IGNORECASE): data['is_gluten_free'] = 1
    if re.search(r'non-?gmo', text, re.IGNORECASE): data['is_non_gmo'] = 1
    if re.search(r'sugar\s*free', text, re.IGNORECASE): data['is_sugar_free'] = 1
    if re.search(r'kosher', text, re.IGNORECASE): data['is_kosher'] = 1
    if re.search(r'organic', text, re.IGNORECASE): data['is_organic'] = 1
    m = re.search(r'Pack of\s*(\d+)', text, re.IGNORECASE)
    if m: data['pack_size'] = safe_float(m.group(1))
    m = re.search(r'([\d\.]+)\s*(?:Ounces|Oz)', text, re.IGNORECASE)
    if m: data['item_size_oz'] = safe_float(m.group(1))

    return data

print("⏳ Re-extracting rich tabular features from catalog_content...")
parsed_train = train['catalog_content'].apply(parse_catalog_content).apply(pd.Series)
parsed_test  = test['catalog_content'].apply(parse_catalog_content).apply(pd.Series)

# Combine original data with parsed features for tabular processing
train_features_tab = pd.concat([train, parsed_train], axis=1)
test_features_tab  = pd.concat([test, parsed_test], axis=1)

print("✅ Rich tabular features re-extracted.")
print("Train features shape before processing:", train_features_tab.shape)
print("Test features shape before processing:", test_features_tab.shape)


# --------------------------
# 1. Define numerical, binary, categorical features
# --------------------------
numerical_features = ['value', 'pack_size', 'item_size_oz']
binary_features = ['is_gluten_free', 'is_non_gmo', 'is_sugar_free', 'is_kosher', 'is_organic']
categorical_features = ['unit']

# --------------------------
# 2. Build transformers
# --------------------------
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

binary_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# --------------------------
# 3. Combine transformers
# --------------------------
preprocessor = ColumnTransformer(transformers=[
    ('num', numerical_transformer, numerical_features),
    ('bin', binary_transformer, binary_features),
    ('cat', categorical_transformer, categorical_features)
])

# --------------------------
# 4. Fit and transform
# --------------------------
X_train_processed = preprocessor.fit_transform(train_features_tab)
X_test_processed  = preprocessor.transform(test_features_tab)

# Ensure dense arrays and save
X_train_processed_dense = X_train_processed.toarray() if hasattr(X_train_processed, 'toarray') else X_train_processed
X_test_processed_dense = X_test_processed.toarray() if hasattr(X_test_processed, 'toarray') else X_test_processed


# --------------------------
# 5. Check shapes
# --------------------------
print("✅ Tabular features processed successfully!")
print("Train shape:", X_train_processed_dense.shape)
print("Test shape:", X_test_processed_dense.shape)

# --------------------------
# 6. Save for reuse in modeling
# --------------------------
# Ensure the save directory exists
save_directory = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline" # Assuming this is the desired save location based on previous cells
os.makedirs(save_directory, exist_ok=True)


train_save_path = os.path.join(save_directory, "train_tabular.npy")
test_save_path = os.path.join(save_directory, "test_tabular.npy")

np.save(train_save_path, X_train_processed_dense, allow_pickle=True)
np.save(test_save_path, X_test_processed_dense, allow_pickle=True)

print(f"✅ Saved tabular features:")
print(f"   → {train_save_path}")
print(f"   → {test_save_path}")

# ============================================================
# 7️⃣ OPTIONAL VALIDATION CHECK
# ============================================================
# Reload to verify shapes before modeling
# Add allow_pickle=True when loading for verification
try:
    train_check = np.load(train_save_path, allow_pickle=True)
    test_check  = np.load(test_save_path, allow_pickle=True)

    print("🔍 Verification:")
    print("Reloaded train_tabular shape:", train_check.shape)
    print("Reloaded test_tabular shape:", test_check.shape)
except Exception as e:
    print(f"❌ Verification failed: {e}")

its output :
# ✅ Raw CSVs loaded successfully for Tabular Feature Engineering!
Train shape: (75000, 4)
Test shape: (75000, 3)
⏳ Re-extracting rich tabular features from catalog_content...
✅ Rich tabular features re-extracted.
Train features shape before processing: (75000, 16)
Test features shape before processing: (75000, 15)
✅ Tabular features processed successfully!
Train shape: (75000, 91)
Test shape: (75000, 91)
✅ Saved tabular features:
   → /content/drive/MyDrive/ColabProjects/MyFeaturePipeline/train_tabular.npy
   → /content/drive/MyDrive/ColabProjects/MyFeaturePipeline/test_tabular.npy
🔍 Verification:
Reloaded train_tabular shape: (75000, 91)
Reloaded test_tabular shape: (75000, 91)

In [ ]:
# if any error occurs :
import numpy as np
import os

# Define the directory where you want to save the file (assuming it's the project directory)
save_directory = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline" # Update this if needed

# Ensure the directory exists
os.makedirs(save_directory, exist_ok=True)

# Assuming X_train_processed_dense is available from a previous cell
# If not, you would need to load or compute it here.
# For example: X_train_processed_dense = np.load("some_path/processed_data.npy", allow_pickle=True)

# Check if X_train_processed_dense exists (from previous cell execution)
if 'X_train_processed_dense' in locals():
    # Define the full path for the file
    train_save_path = os.path.join(save_directory, "train_tabular.npy")

    # Save the array with allow_pickle=True
    np.save(train_save_path, X_train_processed_dense, allow_pickle=True)
    print(f"Saved X_train_processed_dense to {train_save_path} with allow_pickle=True")
else:
    print("X_train_processed_dense not found. Please run the tabular feature engineering cell first.")

# and in separate cell :
import numpy as np
X_train_tab = np.load("/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/train_tabular.npy")
print(X_train_tab.shape)


In [ ]:
# INSTALLING OUR TWO MODELS :
# Install both CatBoost and XGBoost
!pip install --upgrade catboost xgboost


In [ ]:
# BEFORE TRAINING MODEL ;
import numpy as np
import os

# Paths to your embeddings
raw_data_directory = "/content/drive/MyDrive/amazon ML"
text_embedding_directory = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/feature_data"
tabular_embedding_directory = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline"

# Load embeddings
print("📂 Reconstructing X_train, X_test, y_train...")

train = np.load(os.path.join(tabular_embedding_directory, "train_tabular.npy"))
test = np.load(os.path.join(tabular_embedding_directory, "test_tabular.npy"))
train_text = np.load(os.path.join(text_embedding_directory, "train_text_embeddings.npy"))
test_text = np.load(os.path.join(text_embedding_directory, "test_text_embeddings.npy"))
train_img = np.load(os.path.join(raw_data_directory, "img_emb_train_b3.npy"))
test_img = np.load(os.path.join(raw_data_directory, "img_emb_test_b3.npy"))

# Combine them back
X_train = np.hstack([train_text, train, train_img])
X_test = np.hstack([test_text, test, test_img])

# Load target
import pandas as pd
train_df = pd.read_csv(os.path.join(raw_data_directory, "train (1).csv"))
y_train = np.log1p(train_df['price'].values)

# Save combined arrays
np.save("/content/drive/MyDrive/amazon ML/X_train.npy", X_train)
np.save("/content/drive/MyDrive/amazon ML/X_test.npy", X_test)
np.save("/content/drive/MyDrive/amazon ML/y_train.npy", y_train)

print("✅ Saved combined NumPy arrays successfully!")
print(f"Shapes — X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}")


In [ ]:
its output :
📂 Reconstructing X_train, X_test, y_train...
✅ Saved combined NumPy arrays successfully!
Shapes — X_train: (75000, 2011), X_test: (75000, 2011), y_train: (75000,)

In [ ]:
#before training :
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

#before training :
!pip install scikit-learn


In [ ]:
# CATBOOST TRAINING : MODEL-1

import os
import sys
import time
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# ---------- Install if missing (safe, won't reinstall if already present) ----------
def install_if_missing(packages):
    import importlib
    import subprocess
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except Exception:
            print(f"Installing {pkg} ...", flush=True)
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", pkg])

install_if_missing(["catboost", "xgboost"])

from catboost import CatBoostRegressor
import xgboost as xgb

# ============================================================
# 1) Paths & Load data (you said keep paths as-is)
# ============================================================
RAW = "/content/drive/MyDrive/amazon ML"
TEXT_EMB = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline/feature_data"
TAB_EMB = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline"
IMG_EMB = "/content/drive/MyDrive/amazon ML"

train_df = pd.read_csv(os.path.join(RAW, 'train (1).csv'))
test_df  = pd.read_csv(os.path.join(RAW, 'test.csv'))
sample_submission = pd.read_csv(os.path.join(RAW, 'sample_test_out.csv'))

# target (log1p)
y = np.log1p(train_df['price'].values)

# load embeddings (assumes they exist)
X_tab_train = np.load(os.path.join(TAB_EMB, "train_tabular.npy"))
X_tab_test  = np.load(os.path.join(TAB_EMB, "test_tabular.npy"))
X_text_train = np.load(os.path.join(TEXT_EMB, "train_text_embeddings.npy"))
X_text_test  = np.load(os.path.join(TEXT_EMB, "test_text_embeddings.npy"))
X_img_train = np.load(os.path.join(IMG_EMB, "img_emb_train_b3.npy"))
X_img_test  = np.load(os.path.join(IMG_EMB, "img_emb_test_b3.npy"))

X = np.hstack([X_text_train, X_tab_train, X_img_train])
X_test = np.hstack([X_text_test, X_tab_test, X_img_test])

print(f"Data shapes -> X: {X.shape}, X_test: {X_test.shape}", flush=True)

# ============================================================
# 2) Holding out a validation split for iteration-wise SMAPE
# ============================================================
# small holdout (20%) so iteration-wise SMAPE is fast and meaningful
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=42, shuffle=True)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}", flush=True)

# ============================================================
# 3) SMAPE function (use absolute values on original scale)
# ============================================================
def smape(y_true, y_pred):
    # y_true, y_pred expected on original scale (not log)
    num = np.abs(y_pred - y_true)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    return np.mean(num[mask] / denom[mask]) * 100

# helper to print divider
def pr(s=""):
    print(s, flush=True)

# ============================================================
# 4) Training strategy (incremental loop for iteration-wise SMAPE)
#    - CatBoost: train in chunks of `cb_step` iterations
#    - XGBoost: train in chunks of `xgb_step` boosting rounds
# ============================================================
# Settings to keep runtime short but accurate
cat_total_iters = 300
cb_step = 10

xgb_total_rounds = 300
xgb_step = 10

blend_w_cat = 0.8
blend_w_xgb = 0.2

# CatBoost params (fast + robust)
cat_params = {
    'iterations': cb_step,   # we'll call fit repeatedly with init_model
    'learning_rate': 0.05,
    'depth': 6,
    'loss_function': 'MAE',
    'verbose': False,  # we print our own SMAPE
    'thread_count': -1,
    'random_seed': 42,
    'use_best_model': False
}

# XGBoost params
xgb_params = {
    'objective': 'reg:squarederror',
    'learning_rate': 0.05,
    'max_depth': 6,
    'tree_method': 'hist',   # fast on CPU; change to 'gpu_hist' if GPU present & desired
    'verbosity': 0,
    'nthread': -1,
    'seed': 42
}

# ============================================================
# 5) Incremental CatBoost training with SMAPE every cb_step iters
# ============================================================
pr("\n===== CatBoost incremental training (printing val SMAPE every chunk) =====")
cb_model = None
cb_start = time.time()
n_chunks = cat_total_iters // cb_step
for chunk in range(1, n_chunks + 1):
    # If first chunk, create a fresh model; otherwise continue with init_model
    if cb_model is None:
        cb_model = CatBoostRegressor(**cat_params)
        cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50, verbose=False)
    else:
        # continue training by passing the previous model as init_model
        cb_model = CatBoostRegressor(**cat_params)
        cb_model.fit(X_train, y_train, eval_set=(X_val, y_val), init_model=cb_prev_path if 'cb_prev_path' in locals() else cb_model, early_stopping_rounds=50, verbose=False)

    # After each chunk, predict on val and compute SMAPE
    val_preds_cb = cb_model.predict(X_val)
    # predictions are in log space (because we trained on log1p), exponentiate
    val_preds_cb_exp = np.expm1(val_preds_cb)
    y_val_orig = np.expm1(y_val)
    smape_cb = smape(y_val_orig, val_preds_cb_exp)
    pr(f"CatBoost: iteration {chunk*cb_step}/{cat_total_iters}  |  VAL SMAPE: {smape_cb:.4f}%")

    # Save intermediate model to a temp path to allow init_model continuing
    cb_prev_path = os.path.join("/tmp", f"cb_model_chunk_{chunk}.cbm")
    cb_model.save_model(cb_prev_path)

# final catboost predictions on full train and test
cat_oof_preds = cb_model.predict(X)      # log space preds for full train
cat_test_preds = cb_model.predict(X_test)  # log space preds for test

pr(f"CatBoost finished in {time.time() - cb_start:.1f}s\n")

# Save final CatBoost model to Drive
cb_drive_path = os.path.join(RAW, "final_model_catboost.cbm")
cb_model.save_model(cb_drive_path)
pr(f"Saved CatBoost model to: {cb_drive_path}")

CatBoost: iteration 300/300  |  VAL SMAPE: 60.7182%
CatBoost finished in 4060.4s

Saved CatBoost model to: /content/drive/MyDrive/amazon ML/final_model_catboost.cbm

In [ ]:
# in next cell :
# XGBOOST TRAINING
# ====================================================
# ✅ CatBoost + XGBoost Final Blend (0.8 : 0.2)
# ====================================================

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from catboost import CatBoostRegressor

# ====================================================
# ⚙️ SMAPE function
# ====================================================
def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    diff = np.abs(y_true - y_pred) / denom
    diff[denom == 0] = 0.0
    return np.mean(diff) * 100


# ====================================================
# 📂 Load Data
# ====================================================
print("📂 Loading data...")
X_train = np.load("/content/drive/MyDrive/amazon ML/X_train.npy")
y_train = np.load("/content/drive/MyDrive/amazon ML/y_train.npy")
X_test = np.load("/content/drive/MyDrive/amazon ML/X_test.npy")

# ====================================================
# 🐱 Load CatBoost model and regenerate predictions
# ====================================================
print("🐱 Loading CatBoost model and regenerating predictions...")
cat_model = joblib.load("/content/drive/MyDrive/amazon ML/final_model_catboost.pkl")

cat_oof_preds = cat_model.predict(X_train)
cat_test_preds = cat_model.predict(X_test)

# Save regenerated predictions (for record)
np.save("/content/drive/MyDrive/amazon ML/cat_oof_preds.npy", cat_oof_preds)
np.save("/content/drive/MyDrive/amazon ML/cat_test_preds.npy", cat_test_preds)

print("✅ CatBoost predictions regenerated.")


# ====================================================
# 🚀 Train XGBoost model
# ====================================================
print("\n🚀 Training XGBoost (iteration-wise)...")

# Convert to DMatrix for efficiency
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)

xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "mae",
    "tree_method": "hist",  # fastest
    "learning_rate": 0.05,
    "max_depth": 8,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "nthread": -1,
    "seed": 42,
}

num_rounds = 500  # should finish easily under 15 min
watchlist = [(dtrain, "train")]

model_xgb = xgb.train(
    params=xgb_params,
    dtrain=dtrain,
    num_boost_round=num_rounds,
    evals=watchlist,
    verbose_eval=10  # iteration-wise output
)

# Save model
joblib.dump(model_xgb, "/content/drive/MyDrive/amazon ML/final_model_xgboost.pkl")
print("✅ XGBoost model saved successfully.")

# ====================================================
# 🔮 Predictions
# ====================================================
xgb_oof_preds = model_xgb.predict(dtrain)
xgb_test_preds = model_xgb.predict(dtest)

# ====================================================
# 🔀 Blend (0.8 CatBoost + 0.2 XGBoost)
# ====================================================
final_oof = 0.8 * cat_oof_preds + 0.2 * xgb_oof_preds
final_test = 0.8 * cat_test_preds + 0.2 * xgb_test_preds

# ====================================================
# 🧮 Evaluate SMAPE
# ====================================================
final_smape = smape(y_train, final_oof)
print(f"\n🎯 Final Blended SMAPE (0.8 Cat + 0.2 XGB): {final_smape:.4f}%")

# ====================================================
# 💾 Save final submission CSV
# ====================================================
submission = pd.DataFrame({
    "sample_id": np.arange(len(final_test)),
    "price": final_test
})
submission_path = "/content/drive/MyDrive/amazon ML/sample_test_out.csv"
submission.to_csv(submission_path, index=False)

print(f"\n✅ Final submission saved to: {submission_path}")
print("✅ All done!")


output : 📂 Loading data...
 Loading CatBoost model and regenerating predictions...
✅ CatBoost predictions regenerated.

🚀 Training XGBoost (iteration-wise)...
[499]	train-mae:0.25765
✅ XGBoost model saved successfully.

🎯 Final Blended SMAPE (0.8 Cat + 0.2 XGB): 20.3777%

✅ Final submission saved to: /content/drive/MyDrive/amazon ML/sample_test_out.csv
✅ All done!



In [ ]:
✅ XGBoost Booster predictions saved as .npy files
import google.colab.files
import os

# Define the path to the zip file you want to download
# Assuming the zip file was created in the project directory as in the previous cell
project_directory = "/content/drive/MyDrive/ColabProjects/MyFeaturePipeline"
zip_file_path = os.path.join(project_directory, "all_specified_npy_files.zip") # Update filename if different

# Check if the zip file exists before attempting to download
if os.path.exists(zip_file_path):
    print(f"Downloading zip file: {zip_file_path}")
    try:
        google.colab.files.download(zip_file_path)
        print("✅ Download initiated.")
    except Exception as e:
        print(f"❌ Error initiating download: {e}")
        print("You may need to manually download the file from the file browser.")
else:
    print(f"❌ Zip file not found: {zip_file_path}")
    print("Please ensure the zip creation cell was run successfully.")

In [ ]:
# FIXED ERROR IN SAMPLE OUT FILE ;
import numpy as np
import pandas as pd

# ====================================================
# ⚙️ Load model outputs
# ====================================================
# Replace with your actual in-memory variables if available
# These should be the *logged* predictions and targets used during training
# Example: y_train, final_oof, final_test

# If you only have the CSV and no predictions left in memory:
csv_path = "/content/drive/MyDrive/amazon ML/sample_test_out.csv"
submission = pd.read_csv(csv_path)

# ====================================================
# 🧮 Fix if model predicted in log scale
# ====================================================
# If model predictions were log1p-based (common in price regression):
# Reapply expm1 to bring them back to true price scale
if submission["price"].max() < 20:  # heuristic: if all small numbers
    submission["price"] = np.expm1(submission["price"])
    submission["price"] = submission["price"].clip(lower=0)

# ====================================================
# 💾 Save Corrected Submission
# ====================================================
fixed_path = "/content/drive/MyDrive/amazon ML/sample_test_out_fixed.csv"
submission.to_csv(fixed_path, index=False)

print(f"✅ Fixed price values saved to: {fixed_path}")
print(submission.head(10))


output
✅ Fixed price values saved to: /content/drive/MyDrive/amazon ML/sample_test_out_fixed.csv
   sample_id      price
0          0  15.174067
1          1  18.488227
2          2  17.804665
3          3   7.978329
4          4  14.143858
5          5   8.236408
6          6  10.439672
7          7  11.663843
8          8  16.280360
9          9   8.190381

In [ ]:
# COMPUTED CORRECTED SMAPE SCORE A;
import numpy as np

def smape(y_true, y_pred):
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = numerator / denominator
    diff[denominator == 0] = 0
    return np.mean(diff) * 100

# Compute SMAPE on unlogged (real) scale
final_smape = smape(np.expm1(y_train), np.expm1(final_oof))
print(f"🎯 Accurate Final SMAPE (real-scale): {final_smape:.4f}%")



output
smape score is : 51.8239 %